In [ ]:
!apt-get install -y cmake build-essential
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make -j

In [ ]:
# from huggingface_hub import snapshot_download
# model_path = snapshot_download(repo_id="meta-llama/Llama-2-7b-hf", local_dir="llama-2-hf", local_dir_use_symlinks=False)

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    local_dir="tinyllama-hf",
    local_dir_use_symlinks=False
)

In [ ]:
# Inside llama.cpp folder
!python3 convert_hf_to_gguf_update.py ./tinyllama-hf --outfile ./tinyllama-1.1b-chat-q4_0.gguf --outtype q4_0

In [ ]:
!python3 convert_hf_to_gguf.py ./tinyllama-hf --outfile ./tinyllama-1.1b-chat-q4_0.gguf --outtype q8_0

In [ ]:
!./main -m ./tinyllama-1.1b-chat-q4_0.gguf -p "Explain quantization in LLMs" -n 100

In [ ]:
import os
os.getcwd()

In [ ]:
!./main -m ./tinyllama-1.1b-chat-q4_0.gguf -p "Explain quantization in LLMs" -n 100

In [ ]:
!ls -l ./main

In [ ]:
!mkdir -p build

In [ ]:
%cd build

In [ ]:
!cmake ..

In [ ]:
!make

In [ ]:
!./bin/main -m ../tinyllama-1.1b-chat-q4_0.gguf -p "What is quantization in LLMs?" -n 100

In [ ]:
!ls -lh

In [ ]:
!./bin -m ../tinyllama-1.1b-chat-q4_0.gguf -p "Tell me about LLM compression techniques." -n 100

In [ ]:
ls

In [ ]:
os.getcwd()

In [ ]:
!ls -lh bin

In [ ]:
!./bin/llama-cli -m ../tinyllama-1.1b-chat-q4_0.gguf -p "What is quantization in LLMs?" -n 100

✅ llama-cli (official main runner)

✅ llama-run (multi-prompt / batch)

In [ ]:
!pwd

In [ ]:
import os
# !pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ==== Step 1: Load your document ====
def load_documents(file_path):
    print(f"Loading file: {file_path}")
    loader = TextLoader(file_path)
    return loader.load()

# ==== Step 2: Chunk the text ====
def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(documents)

# ==== Step 3: Create or load FAISS VectorStore ====
def create_or_load_faiss(chunks, embedding_model, index_path="faiss_index"):
    if os.path.exists(index_path):
        print("Loading existing FAISS index...")
        # Added allow_dangerous_deserialization=True for security warning
        return FAISS.load_local(index_path, embedding_model, allow_dangerous_deserialization=True)
    print("Creating new FAISS index...")
    db = FAISS.from_documents(chunks, embedding_model)
    db.save_local(index_path)
    return db

# ==== Step 4: Retrieve relevant chunks ====
def get_context_from_query(query, retriever):
    docs = retriever.invoke(query)
    context = "\n\n".join(
            doc.page_content
            for doc in docs
        )

    return context

# ==== Step 5: Build prompt & write to file ====
def build_prompt_file(context, query, prompt_file="prompt.txt"):
    prompt = f"""[INST] <<SYS>>
You are a helpful AI assistant. Use the context to answer the question.
<</SYS>>

Context:
{context}

Question: {query}
Answer: [/INST]
"""
    with open(prompt_file, "w") as f:
        f.write(prompt)
    print(f"Prompt written to {prompt_file}")

# ==== Step 6: Run llama.cpp with GGUF ====
def run_llama_cli(gguf_path, prompt_file="prompt.txt", n_predict=200):
    print("Running inference with llama.cpp...")
    # Corrected path to llama-cli executable
    os.system(f"./build/bin/llama-cli -m {gguf_path} -f {prompt_file} --n-predict {n_predict}")

# ==== === MAIN PIPELINE === ===
def rag_pipeline(
    doc_path="my_notes.txt",
    # Corrected path to GGUF model
    gguf_model_path="./tinyllama-1.1b-chat-q4_0.gguf",
    user_query="What is quantization in LLMs?"
):
    # Load & split
    docs = load_documents(doc_path)
    chunks = chunk_documents(docs)

    # Embeddings
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # FAISS
    vectorstore = create_or_load_faiss(chunks, embedding_model)
    retriever = vectorstore.as_retriever()

    # RAG
    context = get_context_from_query(user_query, retriever)
    build_prompt_file(context, user_query)

    # Inference
    run_llama_cli(gguf_model_path)

# ==== Entry Point ====
if __name__ == "__main__":
    rag_pipeline()


In [ ]:
import os
import subprocess

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


# ==== Step 1: Load your document ====
def load_documents(file_path):
    print(f"Loading file: {file_path}")
    loader = TextLoader(file_path)
    return loader.load()


# ==== Step 2: Chunk the text ====
def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)


# ==== Step 3: Create or load FAISS VectorStore ====
def create_or_load_faiss(chunks, embedding_model, index_path="faiss_index"):
    if os.path.exists(index_path):
        print("Loading existing FAISS index...")
        return FAISS.load_local(
            index_path,
            embedding_model,
            allow_dangerous_deserialization=True
        )

    print("Creating new FAISS index...")
    db = FAISS.from_documents(chunks, embedding_model)
    db.save_local(index_path)
    return db


# ==== Step 4: Retrieve relevant chunks ====
def get_context_from_query(query, retriever):
    docs = retriever.invoke(query)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return context


# ==== Step 5: Build prompt & write to file ====
def build_prompt_file(context, query, prompt_file="prompt.txt"):
    prompt = f"""[INST] <<SYS>>
You are a helpful AI assistant. Use ONLY the given context to answer.

<</SYS>>

Context:
{context}

Question:
{query}

Answer:
[/INST]
"""

    with open(prompt_file, "w", encoding="utf-8") as f:
        f.write(prompt)

    print(f"Prompt written to {prompt_file}")
    return prompt


# ==== Step 6: Run llama.cpp with GGUF ====
# def run_llama_cli(gguf_path, prompt_file="prompt.txt", n_predict=200):
#     print("\nRunning inference with llama.cpp...\n")

#     command = [
#         "./bin/llama-cli",              # Change if your executable is elsewhere
#         "-m", gguf_path,
#         "-f", prompt_file,
#         "-n", str(n_predict)
#     ]

#     result = subprocess.run(
#         command,
#         capture_output=True,
#         text=True
#     )

#     print("=" * 60)
#     print("MODEL OUTPUT")
#     print("=" * 60)
#     print(result.stdout)

#     if result.stderr:
#         print("=" * 60)
#         print("STDERR")
#         print("=" * 60)
#         print(result.stderr)

def run_llama_cli(gguf_path, prompt,  prompt_file="prompt.txt", n_predict=200):
    # command = f"./bin/llama-cli -m {gguf_path} -f {prompt_file} -n {n_predict}"
    # print(command)

    # os.system(command)
    subprocess.run([
    "./bin/llama-cli",
    "-m", gguf_path,
    "-p", prompt,
    "-n", "100"
])


# ==== MAIN PIPELINE ====
def rag_pipeline(
    doc_path="my_notes.txt",
    gguf_model_path="../tinyllama-1.1b-chat-q4_0.gguf",
    user_query="What is quantization in LLMs?"
):

    # Load & split
    docs = load_documents(doc_path)
    chunks = chunk_documents(docs)

    print(f"Total Chunks: {len(chunks)}")

    # Embeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # FAISS
    vectorstore = create_or_load_faiss(
        chunks,
        embedding_model
    )

    retriever = vectorstore.as_retriever()

    # Retrieve Context
    context = get_context_from_query(
        user_query,
        retriever
    )

    print("\nRetrieved Context:\n")
    print(context)

    # Build Prompt
    prompt = build_prompt_file(
        context,
        user_query
    )

    # Run Inference
    run_llama_cli(
        gguf_model_path, prompt
    )


# ==== Entry Point ====
if __name__ == "__main__":
    rag_pipeline()

In [ ]:
# # Remove old FAISS index to ensure a clean re-creation
# !rm -rf faiss_index

In [ ]:
# # Reinstall and update LangChain and related packages to ensure compatibility
# !pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers

GGML (Georgi Gerganov Machine Learning) ek C-based runtime + tensor library hai jo:

Low-level CPU/GPU optimized inference engine hai

Mainly llama.cpp, whisper.cpp, stable-diffusion.cpp jaise projects use karte hain

Original format tha before GGUF came in

No Python dependency – pure C/C++ based

📌 GGUF = file format

📌 GGML = inference engine + tensor library

Practical: Run a GGML-Format LLM Model (like ggml-model-q4.bin)

🧰 Tools:

✅ llama.cpp (same as GGUF)

✅ Prequantized GGML model (e.g., from TheBloke)

✅ main binary from llama.cpp

Step-by-Step GGML Inference

🔹 Step 1: Clone & Build llama.cpp

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp
!make

Step 2: Download a GGML Model

Use any of the prequantized .bin models:

In [ ]:
!wget https://huggingface.co/TheBloke/LLaMa-7B-GGML/resolve/main/ggml-model-q4_0.bin -O ggml-model-q4_0.bin

Step 3: Run Inference

In [ ]:
!./bin/llama-cli \
-m ggml-model-q4_0.bin \
-p "What is quantization in machine learning?" \
-n 100

Prompt: What is quantization in machine learning?

Output: Quantization is the process of reducing the precision of the weights and activations of a neural network. It is commonly used for...

In [ ]:
# %%writefile my_notes.txt
# # Quantization in Large Language Models (LLMs)

# ## 1. Introduction to Quantization

# Quantization is a critical optimization technique in the realm of deep learning, particularly for the deployment of large language models (LLMs). At its core, quantization involves reducing the numerical precision of a model's parameters (weights) and activations from higher-precision formats (e.g., 32-bit floating-point or FP32) to lower-precision formats (e.g., 16-bit floating-point or FP16, 8-bit integers or INT8, or even 4-bit integers or INT4). This reduction in precision has profound implications for a model's efficiency, impacting memory footprint, computational speed, and energy consumption.

# The primary motivation behind quantization stems from the ever-increasing size and complexity of state-of-the-art LLMs. Models like GPT-3, LLaMA, and their successors can have billions or even trillions of parameters, demanding substantial computational resources and memory for both training and inference. Deploying these colossal models on resource-constrained devices, such as mobile phones, edge devices, or even standard GPUs with limited VRAM, becomes a significant challenge without effective compression techniques like quantization.

# ## 2. Why Quantize LLMs?

# Several key factors drive the adoption of quantization for LLMs:

# ### 2.1. Reduced Memory Footprint
# Each parameter in an LLM, when stored in FP32 format, occupies 4 bytes of memory. A model with billions of parameters can quickly consume tens or hundreds of gigabytes of memory. Quantizing these parameters to INT8 (1 byte per parameter) or INT4 (0.5 bytes per parameter) can dramatically reduce the model's memory footprint by 4x or 8x, respectively. This memory reduction is crucial for:
# * **On-device deployment:** Enabling LLMs to run directly on smartphones, IoT devices, and other edge hardware with limited RAM.
# * **Increased batch size:** Allowing larger batch sizes during inference on GPUs, leading to higher throughput.
# * **Reduced model loading time:** Smaller models load faster from disk into memory.

# ### 2.2. Faster Inference Speed
# Processors (CPUs, GPUs, TPUs, NPUs) can often perform arithmetic operations on lower-precision integers much faster than on floating-point numbers. This is due to several reasons:
# * **Specialized hardware:** Many modern AI accelerators and even general-purpose CPUs have dedicated instruction sets (e.g., AVX512 VNNI, Tensor Cores) optimized for integer operations.
# * **Data movement:** Moving smaller data types (INT8 vs. FP32) between memory and compute units requires less bandwidth, which can be a significant bottleneck in deep learning workloads.
# * **Cache efficiency:** More low-precision data can fit into CPU/GPU caches, leading to fewer cache misses and faster access times.
# As a result, quantized LLMs can achieve significantly higher inference speeds, reducing latency for real-time applications.

# ### 2.3. Lower Energy Consumption
# Performing fewer and simpler computations on smaller data types directly translates to lower energy consumption. This is particularly vital for:
# * **Mobile and edge devices:** Extending battery life for AI-powered applications.
# * **Data centers:** Reducing operational costs and environmental impact of large-scale LLM inference.

# ## 3. Types of Quantization Techniques

# Quantization methods can be broadly categorized based on when and how the quantization is performed:

# ### 3.1. Post-Training Quantization (PTQ)
# PTQ involves quantizing a pre-trained, full-precision model without any retraining or fine-tuning. This is often the most straightforward and cost-effective approach. PTQ can be further divided into:
# *   **Dynamic Quantization:** Activations are quantized on-the-fly during inference, while weights are quantized offline. This method requires minimal effort but might not offer the highest accuracy or performance gains as it still has some overhead for activation quantization.
# *   **Static Quantization (Calibration-based PTQ):** Both weights and activations are quantized offline. This requires a small, representative dataset (calibration set) to determine the min/max ranges or histograms of activations. These ranges are then used to set the scaling factors and zero points for quantization. Static PTQ generally yields better accuracy than dynamic quantization and can provide full integer inference benefits.

# ### 3.2. Quantization-Aware Training (QAT)
# QAT involves simulating the effects of quantization during the training process. The model is trained (or fine-tuned) with the quantization operations directly embedded into the computational graph. This allows the model to learn to compensate for the precision loss during training, often leading to higher accuracy compared to PTQ methods, especially for very low bit-widths (e.g., INT4 or INT2). However, QAT requires access to the training data and involves modifying the training pipeline.

# ### 3.3. Mixed-Precision Quantization
# This technique involves using different numerical precisions for different layers or parts of the model. For instance, some layers might be quantized to INT8, while others (e.g., critical layers or those sensitive to precision loss) might remain in FP16 or FP32. This allows for a flexible trade-off between performance/memory savings and accuracy.

# ### 3.4. Hardware-Aware Quantization
# Some quantization techniques are specifically designed to leverage the capabilities of particular hardware accelerators (e.g., GPUs with Tensor Cores, TPUs, or custom ASICs). These methods aim to maximize throughput and efficiency on the target hardware.

# ## 4. Challenges and Considerations in LLM Quantization

# While highly beneficial, quantization for LLMs presents several challenges:

# ### 4.1. Accuracy Drop
# The most significant challenge is maintaining model accuracy. Reducing numerical precision inevitably leads to some information loss, which can manifest as a drop in performance on downstream tasks. This drop is often more pronounced with aggressive quantization (lower bit-widths).

# ### 4.2. Per-Tensor vs. Per-Channel Quantization
# *   **Per-Tensor:** A single scaling factor and zero point are used for all elements within a tensor. Simpler to implement but can be less accurate if the value distribution within the tensor is wide.
# *   **Per-Channel:** Each channel (e.g., output channel of a convolutional layer or rows/columns in a linear layer) gets its own scaling factor and zero point. More complex but often yields better accuracy as it can adapt to varying value distributions across channels.

# ### 4.3. Symmetric vs. Asymmetric Quantization
# *   **Symmetric:** The quantization range is symmetric around zero (e.g., [-127, 127] for INT8). This simplifies calculations but might be suboptimal if the actual value distribution is asymmetric.
# *   **Asymmetric:** The quantization range can be arbitrary (e.g., [min_val, max_val]). This allows for better utilization of the available integer range by aligning it more closely with the actual data distribution.

# ### 4.4. Outlier Handling
# LLMs often exhibit extreme outliers in their weight or activation distributions. These outliers can severely impact quantization accuracy if not handled properly. Techniques like clipping, or using a higher precision for outliers (hybrid quantization), are employed.

# ### 4.5. Data Type Selection
# Choosing the right integer data type (INT8, INT4, INT2) and even floating-point types (FP16, BF16) is crucial. While lower bit-widths offer more benefits, they also pose greater risks to accuracy.

# ### 4.6. Quantization Granularity
# Deciding whether to quantize weights, activations, or both, and at what granularity (layer-wise, block-wise, or global) impacts complexity and performance.

# ## 5. Practical Implementations and Tools

# Several frameworks and libraries provide tools for LLM quantization:

# ### 5.1. NVIDIA TensorRT
# TensorRT is NVIDIA's SDK for high-performance deep learning inference. It supports various quantization schemes, including INT8, and can optimize models for NVIDIA GPUs.

# ### 5.2. PyTorch Quantization
# PyTorch offers built-in quantization utilities, including PTQ (dynamic and static) and QAT, allowing developers to integrate quantization directly into their PyTorch workflows.

# ### 5.3. TensorFlow Lite
# Designed for on-device and edge deployment, TensorFlow Lite provides comprehensive quantization support for TensorFlow models, including integer-only quantization.

# ### 5.4. ONNX Runtime
# ONNX Runtime supports quantization for models in the ONNX format, enabling efficient inference across various hardware backends.

# ### 5.5. llama.cpp and GGUF
# As seen in this context, `llama.cpp` and its GGUF (GGML Universal File Format) are specifically designed for efficient LLM inference on consumer hardware, particularly CPUs. GGUF supports various quantization types (e.g., Q4_0, Q4_1, Q5_0, Q5_1, Q8_0) that balance model size, speed, and quality. The `convert_hf_to_gguf.py` script is used to convert Hugging Face models into the GGUF format with desired quantization levels.

# ## 6. Emerging Trends and Future Directions

# ### 6.1. Even Lower Bit-Widths (INT2, Binary/Ternary Networks)
# Research is continuously pushing the boundaries of quantization, exploring ultra-low bit-widths. While challenging, these could unlock even greater efficiency.

# ### 6.2. Post-Quantization Fine-Tuning
# After initial quantization, a small amount of fine-tuning on a task-specific dataset can often recover significant accuracy, bridging the gap between FP32 and quantized models.

# ### 6.3. Advanced Calibration Techniques
# More sophisticated calibration methods are being developed for PTQ to better estimate activation ranges and improve accuracy without retraining.

# ### 6.4. Quantization for Training
# While most research focuses on inference, quantizing models during training itself is an area of active exploration, potentially speeding up training and reducing memory requirements.

# ### 6.5. Structured vs. Unstructured Sparsity with Quantization
# Combining quantization with pruning/sparsity techniques can lead to even more compact and efficient models. Structured sparsity (e.g., pruning entire channels) is often more hardware-friendly.

# ### 6.6. Dynamic and Adaptive Quantization
# Techniques that can dynamically adjust quantization levels based on input data or model states during inference are gaining interest, offering a potential balance of performance and accuracy.

# ## Conclusion

# Quantization is an indispensable technique for making large language models practical and deployable across a wide range of devices and applications. By intelligently reducing numerical precision, it addresses the critical bottlenecks of memory, speed, and energy consumption. While challenges remain, continuous research and development in quantization techniques are paving the way for more accessible and efficient AI.